# Esame Laboratorio di Programmazione II - 13/01/2026


Scrivete chiaramente sul notebook il vostro nome e matricola e sul nome con cui salvate il file la vostra matricola.

Per ogni funzione e metodo che richiede un campionamento rispetto ad una distribuzione settate il seed a 0: ``np.random.seed(0)``.

Stampate il risultato delle domanda e consegnate il compito eseguito, quindi per cui ogni cella ha il corrispondente output.
Quindi ad esempio

e.g., 
NON scrivete: 
```python 
    lista=np.array([1,2,3])
```
ma 
```python
    lista=np.array([1,2,3]) 
    print('lista =', lista)   
```
Attenzione!Se scrivete:
```python
    lista=np.array([1,2,3])
    lista
```
``lista`` sarà l'unico ouput che si vede di quella cella

NB se l'output è una matrice molto grande non dovete stamparla.

Sarà valutata anche la presentazione, ad esempio un plot senza ettichette sugli assi o illleggibili sarà valutato meno bene di uno con ettichette chiare.

Commentate il compito di modo che si capisca cosa avete fatto.

Controllate attentamente di avere consegnato il file giusto.

In [1]:
#Potete importare direttamente qui le librerie che userete

## Esercizio 1
Un sensore ha registrato le seguenti temperature giornaliere (in °C) durante 12 giorni consecutivi:  
`18, 21, 19, 23, 25, 22, 20, 24, 26, 27, 21, 19`

1. Calcola la temperatura media del periodo.  
2. A causa di una ricalibrazione, tutte le temperature superiori a 22 °C devono essere aumentate di 1 °C. Aggiorna il vettore.  
3. Quanti giorni hanno registrato una temperatura pari o superiore a 25 °C dopo la ricalibrazione?


## Esercizio 2

Un’urna contiene **3 palline rosse** e **2 palline blu**.  
Dopo ogni estrazione la pallina viene **rimessa nell’urna**.

1. Scrivi una funzione che simula **N estrazioni** e restituisce un array NumPy (`1 = rossa`, `0 = blu`).

2. Per valori di N pari a 50, 100, 150, …, 500:
   - ripeti la simulazione **100 volte**
   - calcola la **frequenza relativa media** delle palline rosse.

3. Rappresenta graficamente l’**andamento della frequenza relativa media delle palline rosse** in funzione del numero di estrazioni.


## Esercizio 3

1. Genera una matrice NumPy di dimensioni **6 x 8**, con valori estratti da una distribuzione normale con **media = 2** e **deviazione standard = 3**.

2. Per ogni riga, individua il valore **più vicino a zero** (cioè con valore assoluto minimo). Crea un array con questi **6 valori** (uno per riga).

3. Calcola quante righe hanno come valore più vicino a zero un numero **negativo**.


In [ ]:
# ===========================================================================
# (Setup) creazione dataset `students.csv` (da eseguire una sola volta)
# ===========================================================================

def make_students_csv(filename="students.csv", n=60, seed=42):
    """
    Dataset più "significativo":
    - exam_score cresce (non linearmente) con study_hours e attendance
    - presenza di cluster (studenti regolari / altalenanti / molto bravi)
    - qualche outlier (molte ore ma voto medio, poche ore ma voto alto)
    - qualche valore mancante
    """
    rng = np.random.default_rng(seed)

    student_id = np.arange(1, n + 1)

    group = rng.choice([0, 1, 2], size=n, p=[0.45, 0.40, 0.15])

    study_hours = np.empty(n, dtype=int)
    attendance = np.empty(n, dtype=float)

    idx0 = group == 0
    study_hours[idx0] = rng.integers(12, 28, size=idx0.sum())
    attendance[idx0] = rng.uniform(0.78, 1.00, size=idx0.sum())

    idx1 = group == 1
    study_hours[idx1] = rng.integers(5, 22, size=idx1.sum())
    attendance[idx1] = rng.uniform(0.60, 0.95, size=idx1.sum())

    idx2 = group == 2
    study_hours[idx2] = rng.integers(22, 45, size=idx2.sum())
    attendance[idx2] = rng.uniform(0.85, 1.00, size=idx2.sum())

    attendance = np.round(attendance, 2)

    noise = rng.normal(0, 6, size=n)
    nonlinear_bonus = 8 / (1 + np.exp(-(study_hours - 25) / 4))  # "curva a S"
    exam_score = 35 + 1.4 * study_hours + 22 * (attendance - 0.6) + nonlinear_bonus + noise
    exam_score = np.clip(exam_score, 35, 100)
    exam_score = np.round(exam_score).astype(float)

    for i in rng.choice(np.where(study_hours > 30)[0], size=min(2, np.sum(study_hours > 30)), replace=False):
        exam_score[i] = max(40, exam_score[i] - rng.integers(12, 20))

    for i in rng.choice(np.where(study_hours < 12)[0], size=min(2, np.sum(study_hours < 12)), replace=False):
        exam_score[i] = min(100, exam_score[i] + rng.integers(12, 20))

    p_pass = 1 / (1 + np.exp(-(exam_score - 62) / 6))
    passed = (rng.uniform(0, 1, size=n) < p_pass).astype(int)

    df = pd.DataFrame({
        "student_id": student_id,
        "exam_score": exam_score,
        "study_hours": study_hours,
        "attendance": attendance,
        "passed": passed
    })

    miss_att = rng.choice(df.index, size=max(1, n // 30), replace=False)
    miss_score = rng.choice(df.index.difference(miss_att), size=max(1, n // 30), replace=False)
    df.loc[miss_att, "attendance"] = np.nan
    df.loc[miss_score, "exam_score"] = np.nan

    df.to_csv(filename, index=False)
    return df

df = make_students_csv("students.csv", n=60, seed=0)
print(df.head())


   student_id  exam_score  study_hours  attendance  passed
0           1        68.0           11         NaN       1
1           2        70.0           21        0.99       0
2           3        74.0           18        0.89       1
3           4        82.0           27        0.87       1
4           5        67.0           18        0.85       1


## Esercizio 4

Utilizza il dataset `students.csv`, che contiene le seguenti colonne:  
`student_id`, `exam_score`, `study_hours`, `attendance`, `passed`  
(dove `passed` vale 1 se lo studente ha superato l’esame, 0 altrimenti).

1. Verifica la presenza di **valori mancanti per colonna** ed elimina le righe che ne contengono.

2. Crea una nuova colonna `performance_index` definita come:

   $$
   \texttt{performance\_index} = \texttt{exam\_score} \times \texttt{attendance}
   $$

3. Filtra gli studenti che hanno:
   - `study_hours` > 20  
   - `attendance` > 0.8  

4. Crea un **grafico a barre** con i **10 studenti con performance_index più alto**
   (usa `student_id` come etichetta).

5. Crea uno **scatter plot** con:
   - asse x = `study_hours`
   - asse y = `exam_score`
   - colore dei punti = `performance_index`


## Esercizio 5

Utilizza lo stesso dataset `students.csv`.

1. Determina **numero di righe e colonne** del dataset ed elimina la colonna `passed`.

2. Trova il **valore di `exam_score` più frequente**.

3. Calcola la **media del punteggio d’esame (`exam_score`) per fasce di ore di studio**:
   - `0–10`
   - `11–20`
   - `21–30`
   - `>30`

4. Crea un **boxplot** dei valori di `exam_score` raggruppati per fascia di ore di studio.

5. Crea un **violin plot** della distribuzione di `attendance` per ciascuna fascia di ore di studio.


## Esercizio 6

Genera due array:

`x = np.linspace(0, 5, 80)`  
`y = 2.5 * x + 1.2 + np.random.normal(0, 0.5, 80)`

1. Usa `scipy.optimize.curve_fit` per stimare i parametri `a` e `b` della retta del tipo:

    $$
   y = a \cdot x + b
    $$

2. Plotta i punti originali e la retta ottenuta.

3. Calcola **MAE** (Mean Absolute Error) e **RMSE** (Root Mean Squared Error) tra i valori reali `y` e quelli stimati dal modello, dove:  

   $$
   \text{MAE} = \frac{1}{n}\sum_{i=1}^{n} \lvert y_i - \hat{y}_i \rvert
   $$  

   $$
   \text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n} (y_i - \hat{y}_i)^2}
   $$
